In [1]:
!pip install opencv-python
!pip install ollama

/bin/bash: /home/ubuntu-user/anaconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /home/ubuntu-user/anaconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)


In [2]:
import cv2
import os
import math

def extract_frames(video_path, fps_target=7, output_dir="temp_frames", resize_factor=None):
    """
    Extract exactly fps_target frames per second by seeking to precise timestamps.
    Naming convention: <second>_<frame_in_second>.jpg
    """
    os.makedirs(output_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video file: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    video_fps = cap.get(cv2.CAP_PROP_FPS)
    duration_sec = total_frames / video_fps

    print(f"Video info: {total_frames} frames, {video_fps:.2f} FPS, duration ≈ {duration_sec:.2f} sec")

    saved_count = 0
    max_full_second = math.floor(duration_sec)

    # Loop over each second
    for second_id in range(1, max_full_second + 1):
        for frame_in_second in range(1, fps_target + 1):
            # Target timestamp in seconds
            t = (second_id - 1) + (frame_in_second - 1) / fps_target
            if t > duration_sec:
                break

            # Seek to timestamp (milliseconds)
            cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
            ret, frame = cap.read()
            if not ret:
                continue

            # Resize if requested
            if resize_factor is not None and resize_factor > 0:
                new_w = int(frame.shape[1] * resize_factor)
                new_h = int(frame.shape[0] * resize_factor)
                frame = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA)

            filename = f"{second_id}_{frame_in_second}.jpg"
            filepath = os.path.join(output_dir, filename)
            cv2.imwrite(filepath, frame)
            saved_count += 1

            if saved_count % 100 == 0:
                print(f"Saved {saved_count} frames...")

    cap.release()

    expected_frames = max_full_second * fps_target
    print(f"✅ Extraction complete. Saved {saved_count} frames to '{output_dir}'")
    print(f"Expected ≈ {expected_frames} frames (plus possible partial second).")


In [3]:
video_file = '/home/ubuntu-user/Desktop/temp_video/videos/1B.avi'#'/media/ubuntu-user/KINGSTON/00Workspace_portable/videos/1B.avi'
#cp /media/ubuntu-user/KINGSTON/00Workspace_portable/videos/1B.avi /home/ubuntu-user/Desktop/temp_video/videos/1B.avi
fps_rate = 7
temp_folder = '/home/ubuntu-user/Desktop/portable/demo_temp_frames'

# Extract frames resized to 50% width & height
extract_frames(video_file, fps_target=fps_rate, output_dir=temp_folder, resize_factor=0.5)

Video info: 2196 frames, 7.00 FPS, duration ≈ 313.71 sec
Saved 100 frames...
Saved 200 frames...
Saved 300 frames...
Saved 400 frames...
Saved 500 frames...
Saved 600 frames...
Saved 700 frames...
Saved 800 frames...
Saved 900 frames...
Saved 1000 frames...
Saved 1100 frames...
Saved 1200 frames...
Saved 1300 frames...
Saved 1400 frames...
Saved 1500 frames...
Saved 1600 frames...
Saved 1700 frames...
Saved 1800 frames...
Saved 1900 frames...
Saved 2000 frames...
Saved 2100 frames...
✅ Extraction complete. Saved 2191 frames to '/home/ubuntu-user/Desktop/portable/demo_temp_frames'
Expected ≈ 2191 frames (plus possible partial second).


In [ ]:
#watch -n 1 nvidia-smi

In [5]:
import ollama

response = ollama.chat(model='qwen3-vl:8b', messages=[
    {'role': 'user', 'content': 'Hello Qwen, what is 2+2?'}
])

print(response['message']['content'])




Hello! 2 + 2 equals **4**. 😊 Let me know if you have any other questions—I'm here to help!


In [ ]:
import ollama

# 1. 改 Model Name
MODEL_NAME = 'qwen3-vl:32b'

print(f"正在啟動 {MODEL_NAME}，呢個過程可能會比 8B 慢少少...")

try:
    response = ollama.chat(model=MODEL_NAME, messages=[
        {'role': 'user', 'content': 'Hello Qwen, please confirm your model size and tell me what is 2+2?'}
    ])
    print("--- 回覆內容 ---")
    print(response['message']['content'])
    print("---------------")
    print("✅ 測試成功！32B 模型運作正常。")
except Exception as e:
    print(f"❌ 啟動失敗：{e}")
    print("提示：如果報錯係 'out of memory'，代表 VRAM 爆咗。")

In [6]:
import ollama
import glob

# Path to your frames
frame_dir = '/home/ubuntu-user/Desktop/portable/demo_temp_frames'

# Collect first 10 frames (adjust extension if needed: .jpg, .png, etc.)
frames = sorted(glob.glob(f"{frame_dir}/*.jpg"))[:70] #70 for 10 seconds

# Build the message with images attached
messages = [
    {
        'role': 'user',
        'content': r'''For this task, you are an expert at identifying the behaviors of stentors, a type of unicellular ciliated protist. 
        The uploaded video shows a stentor under magnification, which may be engaging in one or more typical behaviors. 
        The video consists of subsampled frames. Given the evidence in the video, identify whether the stentor “contracts.” 
        The definition of a “contraction” is any behavior in which the stentor changes from an extended shape resembling a trumpet to a condensed, ball-like shape. 
        
        Answer “yes” if the stentor contracts at any point in the video; answer “no” if it does not. Do not provide any answer other than “yes” or “no” (such as “maybe,” “I’m not sure,” etc.). In addition to your main answer, give a brief explanation for why you did or did not observe a contraction in the video. 
        
        meanwhile please provide an integer contraction score for your judgement, where 0 for not a contraction and 100 for a sure contraction, so a score in the middle like 50 represents a hard case.
        response format: "[yes or no]/[contraction score]/[reasoning]" e.g. "yes/87/At 0.2 s, the stentor is fully......... Therefore, contraction occurs....."
        warning: must response in english regardless of my location, timezone and system/browser language settings. 
        
        When classifying stentor behaviors, keep in mind the following important points:
        
        -) Most contractions happen quickly (often taking just one subsampled frame). Some contractions, however, are slower and may take ten or more frames. “Slow” contractions should still be annotated as contractions. 
        -) Not all contractions result in a fully ball-like shape. Some contractions involve a clear and measurable shortening of the stentor’s body length, where the distance between the “head” and “tail” decreases noticeably but the organism does not become spherical. These partial or incomplete contractions should still be annotated as contractions. 
        -) The video may show objects other than the stentor, such as algae, plastic beads, glass needles, or similar debris. You should ignore these objects and focus only on the stentor. 
        Besides, the coming video associated with this task should be evaluated independently based solely on its own visual evidence. If additional videos follow, each video should be treated as a separate and independent task. Do not use information, observations, or conclusions from previous videos when evaluating later videos.''',
        'images': frames   # <-- images go inside the message dict
    }
]

# Send to Ollama
response = ollama.chat(model='qwen3-vl:8b', messages=messages)

print(response['message']['content'])


yes/85/The stentor shows a measurable shortening of its body length, transitioning from an extended trumpet-like shape to a more condensed form across the frames. This shortening constitutes a contraction as defined, even if not fully spherical. The change is evident in multiple frames, with the organism reducing its distance between head and tail.


In [7]:
print(response)


model='qwen3-vl:8b' created_at='2026-04-13T07:56:07.480832156Z' done=True done_reason='stop' total_duration=17379667793 load_duration=1907614947 prompt_eval_count=23093 prompt_eval_duration=7090080390 eval_count=650 eval_duration=6383313053 message=Message(role='assistant', content='yes/85/The stentor shows a measurable shortening of its body length, transitioning from an extended trumpet-like shape to a more condensed form across the frames. This shortening constitutes a contraction as defined, even if not fully spherical. The change is evident in multiple frames, with the organism reducing its distance between head and tail.', thinking="Got it, let's analyze the images. The task is to check if the stentor contracts. A contraction is when it goes from extended (trumpet-like) to condensed (ball-like) or shorter. Looking at the sequence of frames, I need to see if there's a change in shape.\n\nFirst, observe the stentor's form. In the initial frames, it's a long, trumpet-like structure.

In [48]:
import os
import glob
import json
import ollama

def process_video_clips(
    frame_dir,
    clip_length_sec=10,
    fps_target=7,
    output_json="/home/ubuntu-user/Desktop/portable/video_analysis.json",
    prompt=None,
    model="qwen3-vl:8b"
):
    """
    Process video frames in fixed-length clips and query LLM for each clip.
    Results are saved as JSON with clip ranges as keys and message content as values.
    """
    if prompt is None:
        prompt = "Describe what happens in these frames"

    # Load existing results safely
    results = {}
    if os.path.exists(output_json):
        try:
            with open(output_json, "r") as f:
                results = json.load(f)
        except (json.JSONDecodeError, ValueError):
            print(f"⚠ Warning: {output_json} is empty or invalid. Starting fresh.")
            results = {}

    # Collect all frames
    frames = sorted(glob.glob(os.path.join(frame_dir, "*.jpg")))

    # Group frames by second
    frames_by_second = {}
    for f in frames:
        fname = os.path.basename(f)
        second = int(fname.split("_")[0])  # e.g., "4_3.jpg" → second=4
        frames_by_second.setdefault(second, []).append(f)

    if not frames_by_second:
        print("⚠ No frames found in directory.")
        return

    max_second = max(frames_by_second.keys())

    # Iterate over clips
    for start_sec in range(1, max_second + 1, clip_length_sec):
        end_sec = min(start_sec + clip_length_sec - 1, max_second)
        clip_key = f"{start_sec}-{end_sec}"

        # Skip if already processed
        if clip_key in results:
            print(f"⏩ Skipping clip {clip_key}, already processed.")
            continue

        # Collect frames for this clip
        clip_frames = []
        for sec in range(start_sec, end_sec + 1):
            clip_frames.extend(frames_by_second.get(sec, []))

        if not clip_frames:
            continue

        # Build message
        messages = [
            {
                "role": "user",
                "content": prompt,
                "images": clip_frames
            }
        ]

        # Query Ollama
        print(f"▶ Processing clip {clip_key} with {len(clip_frames)} frames...")
        response = ollama.chat(model=model, messages=messages)

        # Save only the message content (JSON-serializable)
        results[clip_key] = response['message']['content']

        # Write JSON immediately
        with open(output_json, "w") as f:
            json.dump(results, f, indent=2)

        print(f"✅ Finished clip {clip_key}, saved to {output_json}")

    print("🎉 All clips processed.")


In [49]:
frame_dir = '/home/ubuntu-user/Desktop/portable/demo_temp_frames'
output_json = '/home/ubuntu-user/Desktop/portable/1B_video_analysis.json'

prompt = r"""You are good at identifying behaviors of stentors from video.
The uploaded video is about a stentor. The video consists of subsampled frames.
Tell me whether the stentor exhibits the "contraction" behavior, which means
that the Stentor changes from a "trumpet" (extended) shape to a "droplet"
(contracted) shape. Note that the contraction behavior can be quick, especially
in the subsampled video. You need to answer "yes" if the stentor exhibits
"contraction"; otherwise, "no". Moreover, you should explain when the
"contraction" happens."""

process_video_clips(
    frame_dir=frame_dir,
    clip_length_sec=10,
    fps_target=7,
    output_json=output_json,
    prompt=prompt,
    model="qwen3-vl:8b"
)


⚠ Warning: /home/ubuntu-user/Desktop/portable/1B_video_analysis.json is empty or invalid. Starting fresh.
▶ Processing clip 1-10 with 70 frames...
✅ Finished clip 1-10, saved to /home/ubuntu-user/Desktop/portable/1B_video_analysis.json
▶ Processing clip 11-20 with 70 frames...
✅ Finished clip 11-20, saved to /home/ubuntu-user/Desktop/portable/1B_video_analysis.json
▶ Processing clip 21-30 with 70 frames...
✅ Finished clip 21-30, saved to /home/ubuntu-user/Desktop/portable/1B_video_analysis.json
▶ Processing clip 31-40 with 70 frames...
✅ Finished clip 31-40, saved to /home/ubuntu-user/Desktop/portable/1B_video_analysis.json
▶ Processing clip 41-50 with 70 frames...
✅ Finished clip 41-50, saved to /home/ubuntu-user/Desktop/portable/1B_video_analysis.json
▶ Processing clip 51-60 with 70 frames...
✅ Finished clip 51-60, saved to /home/ubuntu-user/Desktop/portable/1B_video_analysis.json
▶ Processing clip 61-70 with 70 frames...
✅ Finished clip 61-70, saved to /home/ubuntu-user/Desktop/port

In [ ]:
!rsync -av -progress /media/ubuntu-user/KINGSTON/00Workspace_portable/bot_clips/selected_nomove/ /home/ubuntu-user/Desktop/temp_video/selected_nomove/

In [2]:
!mkdir /home/ubuntu-user/Desktop/temp_video/

/bin/bash: /home/ubuntu-user/anaconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)


In [3]:
!rsync -av /media/ubuntu-user/KINGSTON/00Workspace_portable/bot_clips/seg/ /home/ubuntu-user/Desktop/temp_video/seg/

/bin/bash: /home/ubuntu-user/anaconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
sending incremental file list
created directory /home/ubuntu-user/Desktop/temp_video/seg
./
alteration/
alteration/11A_0146.mp4
alteration/11B_0054_B_P.mp4
alteration/12A_0130_P.mp4
alteration/13A_0144_P.mp4
alteration/13A_1321.mp4
alteration/13A_1725_C.mp4
alteration/14A_0030_P.mp4
alteration/15A_0053_C_P.mp4
alteration/15A_0340.mp4
alteration/15B_0209_C_P.mp4
alteration/15B_1047_C.mp4
alteration/15B_1529_P.mp4
alteration/17B_0005_C_P.mp4
alteration/17D_0009_C.mp4
alteration/17E_0028_B.mp4
alteration/17F_0045.mp4
alteration/17F_0305.mp4
alteration/17G_0035_C.mp4
alteration/17H_0101.mp4
alteration/1A_0138.mp4
alteration/1B_0040.mp4
alteration/1C_0545_C.mp4
alteration/1C_0900.mp4
alteration/1E_0026_B.mp4
alteration/1F_0005_P.mp4
alteration/3B_0045_C.mp4
alteration/3C_0041_B_C_P.mp4
alteration/3D_0039_C.mp4
alteration/4B_0110_B_C.mp4
alteration/4D_0035.mp4
alteration/4D_0129

In [ ]:
import os
import cv2
import math
import shutil
import pandas as pd
import ollama

# --- CONFIGURATION ---

INPUT_DIR = '/home/ubuntu-user/Desktop/temp_video/selected_nomove'
OUTPUT_CSV = '/home/ubuntu-user/Desktop/portable/n_stentor_analysis_results.csv'

INPUT_DIR = '/home/ubuntu-user/Desktop/temp_video/seg/contraction_ub'
OUTPUT_CSV = '/home/ubuntu-user/Desktop/portable/c_stentor_analysis_results.csv'

INPUT_DIR = '/home/ubuntu-user/Desktop/temp_video/seg/alteration'
OUTPUT_CSV = '/home/ubuntu-user/Desktop/portable/a_stentor_analysis_results.csv'

INPUT_DIR = '/home/ubuntu-user/Desktop/temp_video/seg/bending'
OUTPUT_CSV = '/home/ubuntu-user/Desktop/portable/b_stentor_analysis_results.csv'
'''
INPUT_DIR = '/home/ubuntu-user/Desktop/temp_video/seg/detachment'
OUTPUT_CSV = '/home/ubuntu-user/Desktop/portable/d_stentor_analysis_results.csv'


'''
TEMP_FRAME_DIR = '/home/ubuntu-user/Desktop/portable/temp_processing_frames'
MODEL_NAME = 'qwen3-vl:8b'
MODEL_NAME = 'qwen3-vl:32b'

FPS_RATE = 7

# --- HELPER FUNCTIONS ---

def extract_all_frames(video_path, fps_target=7, output_dir="temp_frames"):
    """Extracts frames from the entire 10s video and returns list of paths."""
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return []

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    video_fps = cap.get(cv2.CAP_PROP_FPS)
    duration_sec = total_frames / video_fps

    frame_paths = []
    # Loop through the short video (max 10s)
    # We use a simple count-based seek for these short clips
    for i in range(int(duration_sec * fps_target)):
        t = i / fps_target
        cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
        ret, frame = cap.read()
        if not ret:
            break
        
        # Optional: Resize to 50% for faster Ollama processing
        frame = cv2.resize(frame, (0,0), fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)
        
        file_path = os.path.join(output_dir, f"frame_{i:03d}.jpg")
        cv2.imwrite(file_path, frame)
        frame_paths.append(file_path)

    cap.release()
    return frame_paths

# --- MAIN PIPELINE ---

# 1. Load existing results to skip processed videos
processed_files = set()
if os.path.exists(OUTPUT_CSV):
    df_existing = pd.read_csv(OUTPUT_CSV)
    processed_files = set(df_existing['filename'].tolist())

# 2. Identify video files
valid_extensions = ('.mp4', '.avi', '.mov', '.mkv')
all_videos = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(valid_extensions)]

results = []

print(f"Found {len(all_videos)} videos. {len(processed_files)} already processed.")

for video_name in all_videos:
    if video_name in processed_files:
        print(f"Skipping: {video_name}")
        continue

    video_path = os.path.join(INPUT_DIR, video_name)
    print(f"Processing: {video_name}...")

    try:
        # A. Extract Frames
        frames = extract_all_frames(video_path, fps_target=FPS_RATE, output_dir=TEMP_FRAME_DIR)
        
        if not frames:
            print(f"Warning: Could not extract frames for {video_name}")
            continue

        # B. Prepare Ollama Request
        prompt_text = r"""For this task, you are an expert at identifying the behaviors of stentors, a type of unicellular ciliated protist. 
The uploaded video shows a stentor under magnification, which may be engaging in one or more typical behaviors. 
The video consists of subsampled frames. Given the evidence in the video, identify whether the stentor “contracts.” 
The definition of a “contraction” is any behavior in which the stentor changes from an extended shape resembling a trumpet to a condensed, ball-like shape. 

Answer “yes” if the stentor contracts at any point in the video; answer “no” if it does not. Do not provide any answer other than “yes” or “no” (such as “maybe,” “I’m not sure,” etc.). In addition to your main answer, give a brief explanation for why you did or did not observe a contraction in the video. 

meanwhile please provide an integer contraction score for your judgement, where 0 for not a contraction and 100 for a sure contraction, so a score in the middle like 50 represents a hard case.
response format: "[yes or no]/[contraction score]/[reasoning]" e.g. "yes/87/At 0.2 s, the stentor is fully......... Therefore, contraction occurs....."
warning: must response in english regardless of my location, timezone and system/browser language settings. 

When classifying stentor behaviors, keep in mind the following important points:
-) Most contractions happen quickly (often taking just one subsampled frame). Some contractions, however, are slower and may take ten or more frames. “Slow” contractions should still be annotated as contractions. 
-) Not all contractions result in a fully ball-like shape. Some contractions involve a clear and measurable shortening of the stentor’s body length, where the distance between the “head” and “tail” decreases noticeably but the organism does not become spherical. These partial or incomplete contractions should still be annotated as contractions. 
-) The video may show objects other than the stentor, such as algae, plastic beads, glass needles, or similar debris. You should ignore these objects and focus only on the stentor. 
Besides, the coming video associated with this task should be evaluated independently based solely on its own visual evidence. If additional videos follow, each video should be treated as a separate and independent task. Do not use information, observations, or conclusions from previous videos when evaluating later videos."""

        # C. Call Ollama
        response = ollama.chat(
            model=MODEL_NAME,
            messages=[{
                'role': 'user',
                'content': prompt_text,
                'images': frames
            }]
        )

        ai_response = response['message']['content']

        # D. Save Result to List
        new_row = {'filename': video_name, 'response': ai_response}
        results.append(new_row)

        # E. Append to CSV immediately (to prevent data loss if script stops)
        pd.DataFrame([new_row]).to_csv(OUTPUT_CSV, mode='a', index=False, header=not os.path.exists(OUTPUT_CSV))

        # F. Cleanup Frames
        shutil.rmtree(TEMP_FRAME_DIR)
        print(f"Done: {video_name}")

    except Exception as e:
        print(f"Error processing {video_name}: {e}")

print(f"All done! Results saved to {OUTPUT_CSV}")

Found 16 videos. 3 already processed.
Skipping: 3C_0042_A_C_P.mp4
Skipping: 15B_1536.mp4
Skipping: 1F_0014.mp4
Processing: 17E_0030_A.mp4...
Done: 17E_0030_A.mp4
Processing: 1B_0140.mp4...


In [ ]:
# 試吓有無 32b 版本（如果官方已釋出）
~/ollama/bin/ollama pull qwen3-vl:32b

~/ollama/bin/ollama list


# 建立桌面暫存資料夾
mkdir -p ~/Desktop/checkpoints

# 將 Ollama 的模型庫全部複製過去
# 呢度包含咗 manifests (索引) 同 blobs (數據內容)
cp -r ~/.ollama/models ~/Desktop/checkpoints/

rsync -av ~/.ollama/models/ /media/ubuntu-user/KINGSTON/portable/QwenCHKPs/models/


rsync -r ~/Desktop/checkpoints/ /media/ubuntu-user/KINGSTON/portable/QwenCHKPs/


# 喺新機執行：
mkdir -p ~/.ollama
cp -r /media/ubuntu-user/KINGSTON/portable/QwenCHKPs/models/* ~/.ollama/